Ingestamos en bronce la cotización de los instrumentos del día. 

Se usa yfinance cómo fuente de información ya qué es una de los proveedores más conocidos y universal lo cuál lo hace ideal para un primer prototipo.
Scope: solo tomamos la data si esta presente en dicha fuente, caso contrario no será ingestada, pero tampoco generará inconvenientes en el pipeline. Esta fuente no incluye información de mercado bonos soberanos de Argentina, qué deberian ser traidos de un proveedor alternativo.

In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Tabla donde se enlistan todos los tickets, esta tabla se generó en base a todos los simbolos de interés de los cuáles se tiene infomración de transaccciones.

In [0]:
ruta_origen = 'iol_challenge.bronze.tickets'

In [0]:
df = (
        spark.read 
        .format("delta") 
        .table(ruta_origen)
)

Agregamos el sufijo .BA qué es la denominación de yfinance cuando se trabaja con un activo del mercado argentino.

In [0]:
tickets = [row["simbolo_titulo"] + ".BA" for row in df.select("simbolo_titulo").collect()]

In [0]:
tickets[0]

'IBM.BA'

In [0]:
import yfinance as yf

Cargamos la api con una excepción en caso de limit requests. La liberia yf por defecto hace aumaticamente manejo de las excepciones en caso de información no encontrada, limites de requests y nos garantiza qué si hay datos qué no se logran cargar entonces no los obtendremos pero tampoco interrumpirá el proceso.
Agregamos de forma defensiva un bloque try para capturar fallas de la libreria requests en caso de errores de errores HTTP.

In [0]:
import requests

try:
    data_raw = yf.download(
        tickets
        ,start="2026-01-01"
        ,interval="1d"
    )
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 429:
        print("Rate limit alcanzado (HTTP 429). Es necesario aplicar backoff.")
    else:
        print(f"Error HTTP: {e}")
except requests.exceptions.RequestException as e:
    print(f"Error de red o conexión: {e}")

[                       0%                       ]  3 of 1445 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MGCQO.BA"}}}
$MGCQO.BA: possibly delisted; no timezone found
$IRCLD.BA: possibly delisted; no timezone found
[                       1%                       ]  8 of 1445 completed$TX31.BA: possibly delisted; no timezone found
[                       1%                       ]  8 of 1445 completed$TVPY.BA: possibly delisted; no timezone found
[                       1%                       ]  11 of 1445 completed$GFGC77262A.BA: possibly delisted; no timezone found
[                       1%                       ]  14 of 1445 completed$GFGC11777F.BA: possibly delisted; no timezone found
[                       1%                       ]  15 of 1445 completed$GFGC67747A.BA: possibly delisted; no timezone found
[                       1%                       ]  16 of 1445 completed$GFGC91761F.BA: pos

Una vez qué tenemos la información de yfinance cargada tenemos qué llevarla mediante operaciones de pivot y unpivot de spark hacia el formato qué necesitamos, el de una tabla con primary key date e instrumento y con columnas las aperturas, clausuras, máximos, minimos y volumenes transaccionados.
Es una buena practica hacer este procesamiento utilizando spark dataframes ya qué en el caso de qué el volumen de datos escale los issues de performance podrán ser mejor resueltos.

In [0]:
import pyspark.sql.functions as F

In [0]:
data = data_raw.reset_index(level=0)
data.columns = [ (col[0], col[1].replace(".BA", "").replace(".","-")) for col in data.columns]

In [0]:
df = spark.createDataFrame(data)

In [0]:
df.columns

["('Date', '')",
 "('Adj Close', '#MAV150560106')",
 "('Adj Close', 'AE38')",
 "('Adj Close', 'AE38C')",
 "('Adj Close', 'AE38D')",
 "('Adj Close', 'AER9O')",
 "('Adj Close', 'AERBD')",
 "('Adj Close', 'AERBO')",
 "('Adj Close', 'AKO-B')",
 "('Adj Close', 'AL29')",
 "('Adj Close', 'AL29C')",
 "('Adj Close', 'AL29D')",
 "('Adj Close', 'AL30')",
 "('Adj Close', 'AL30C')",
 "('Adj Close', 'AL30D')",
 "('Adj Close', 'AL35')",
 "('Adj Close', 'AL35C')",
 "('Adj Close', 'AL35D')",
 "('Adj Close', 'AL41')",
 "('Adj Close', 'AL41C')",
 "('Adj Close', 'AL41D')",
 "('Adj Close', 'ALUC1000JU')",
 "('Adj Close', 'ALUC1047AB')",
 "('Adj Close', 'ALUC1047FE')",
 "('Adj Close', 'ALUC1097FE')",
 "('Adj Close', 'ALUC1150AB')",
 "('Adj Close', 'ALUC1200AB')",
 "('Adj Close', 'ALUC1200JU')",
 "('Adj Close', 'ALUC1300JU')",
 "('Adj Close', 'ALUC39704F')",
 "('Adj Close', 'ALUC84704A')",
 "('Adj Close', 'ALUC99704F')",
 "('Adj Close', 'ALUV1047AB')",
 "('Adj Close', 'ALUV79704A')",
 "('Adj Close', 'ALUV897

In [0]:
columnas_fijas = ["('Date', '')"]
columnas_variables = [col for col in df.columns if col not in columnas_fijas]

In [0]:
columnas_variables

["('Adj Close', '#MAV150560106')",
 "('Adj Close', 'AE38')",
 "('Adj Close', 'AE38C')",
 "('Adj Close', 'AE38D')",
 "('Adj Close', 'AER9O')",
 "('Adj Close', 'AERBD')",
 "('Adj Close', 'AERBO')",
 "('Adj Close', 'AKO-B')",
 "('Adj Close', 'AL29')",
 "('Adj Close', 'AL29C')",
 "('Adj Close', 'AL29D')",
 "('Adj Close', 'AL30')",
 "('Adj Close', 'AL30C')",
 "('Adj Close', 'AL30D')",
 "('Adj Close', 'AL35')",
 "('Adj Close', 'AL35C')",
 "('Adj Close', 'AL35D')",
 "('Adj Close', 'AL41')",
 "('Adj Close', 'AL41C')",
 "('Adj Close', 'AL41D')",
 "('Adj Close', 'ALUC1000JU')",
 "('Adj Close', 'ALUC1047AB')",
 "('Adj Close', 'ALUC1047FE')",
 "('Adj Close', 'ALUC1097FE')",
 "('Adj Close', 'ALUC1150AB')",
 "('Adj Close', 'ALUC1200AB')",
 "('Adj Close', 'ALUC1200JU')",
 "('Adj Close', 'ALUC1300JU')",
 "('Adj Close', 'ALUC39704F')",
 "('Adj Close', 'ALUC84704A')",
 "('Adj Close', 'ALUC99704F')",
 "('Adj Close', 'ALUV1047AB')",
 "('Adj Close', 'ALUV79704A')",
 "('Adj Close', 'ALUV89704F')",
 "('Adj C

In [0]:
df_unpivoted = df.unpivot(
    ids=columnas_fijas,
    values=columnas_variables,
    variableColumnName="ticker",
    valueColumnName="value"
)

In [0]:
df_unpivoted.limit(100).display()

"('Date', '')",ticker,value
2026-01-02T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-05T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-06T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-07T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-08T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-09T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-12T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-13T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-14T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null
2026-01-15T00:00:00.000Z,"('Adj Close', '#MAV150560106')",null


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import regexp_replace

df_with_tickers = (
    df_unpivoted
    .withColumn("ticker_limpio",  regexp_replace(F.col("ticker"), r"[()'\"]", "") )
    .withColumn("tipo", F.trim(F.split(F.col("ticker_limpio"), ",")[0]))
    .withColumn("simbolo", F.trim(F.split(F.col("ticker_limpio"), ",")[1]))
    .withColumn("date", F.col("('Date', '')").cast("date"))
    .filter(F.col("tipo").isin("Close", "Open","High", "Low", "Volume")) # Solo tomamos F.columnas elementales de la cotizacion
    .filter(F.col("value").isNotNull()) # Solo tomamos tickets con información disponible
    .select(
        F.col("date"), 
        F.col("simbolo"), 
        F.col("tipo"), 
        F.col("value")
    )
)


In [0]:
df_with_tickers.limit(30).display()

date,simbolo,tipo,value
2026-01-02,A3,Close,2093.180908203125
2026-01-05,A3,Close,2093.180908203125
2026-01-06,A3,Close,2093.180908203125
2026-01-07,A3,Close,2093.180908203125
2026-01-08,A3,Close,2093.180908203125
2026-01-09,A3,Close,2093.180908203125
2026-01-12,A3,Close,2093.180908203125
2026-01-13,A3,Close,2093.180908203125
2026-01-14,A3,Close,2093.180908203125
2026-01-15,A3,Close,2093.180908203125


In [0]:
df_pivoted = df_with_tickers.groupBy("date", "simbolo").pivot(
    "tipo"
).agg(F.first("value"))

In [0]:
df_pivoted.limit(10).display()

date,simbolo,Close,High,Low,Open,Volume
2026-01-02,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-05,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-06,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-07,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-08,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-09,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-12,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-13,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-14,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0
2026-01-15,A3,2093.180908203125,2093.180908203125,2093.180908203125,2093.180908203125,0.0


Con la data en el formato de una fila por date y por simbolo completamos un append de la data raw de los instrumentos en el modelo de bronce. Además agregamos timestamp de ejecución, particionamiento y un control de calidad a la información.

In [0]:
target_path = "iol_challenge.bronze.raw_ingestion"

In [0]:
(
    df_pivoted
        .withColumn("data_type", F.lit("instrument"))
        .withColumn("data", F.to_json(F.struct(*df_pivoted.columns), options={"ignoreNullFields": "true"}))
        .withColumn("anio", F.year(F.col("date")))
        .withColumn("mes", F.month(F.col("date")))
        .withColumn("dia", F.month(F.col("date")))
        .withColumn("timestamp_ejecucion", F.now())
        .withColumn("errores_calidad", F.array_remove(
                F.array(
                    F.when(F.col("simbolo").isNull(), "SIMBOLO_NULO"),
                    F.when(F.length(F.col("Close")) < 0, "VALOR_DE_CIERRE_NEGATIVO"),
                    F.when(F.length(F.col("High")) < 0, "VALOR_MAXIMO_NEGATIVO"),
                    F.when(F.length(F.col("Low")) < 0, "VALOR_MINIMO_NEGATIVO"),
                    F.when(F.length(F.col("Open")) < 0, "VALOR_DE_APERTURA_NEGATIVO"),
                    F.when(F.length(F.col("Volume")) < 0, "VOLUMEN_DE_TRANSACCION_NEGATIVO"),
                ),
                None 
            )
        ).withColumn("tiene_errores_calidad",
            F.col("errores_calidad").isNotNull()
        ).select(
            F.col("data_type"),
            F.col("data"),
            F.col("dia"),
            F.col("anio"),
            F.col("mes"),
            F.col("timestamp_ejecucion"),
            F.col("errores_calidad"),
            F.col("tiene_errores_calidad"),
        )
        .write
        .format("delta")      
        .mode("append")
        .partitionBy("anio", "mes", "dia")
        .saveAsTable(target_path)
)

Validamos cuantos datos de cada fuente se han ingestado en bronce

In [0]:
%sql
select count(*), data_type FROM iol_challenge.bronze.raw_ingestion GROUP BY data_type;

count(*),data_type
100000,transaction
83093,instrument


Validamos algunos de los datos ingestados

In [0]:
%sql
SELECT * FROM iol_challenge.bronze.raw_ingestion WHERE data_type = 'instrument' LIMIT 10;

data_type,data,dia,anio,mes,timestamp_ejecucion,errores_calidad,tiene_errores_calidad
instrument,"{""date"":""2026-01-26"",""simbolo"":""ADBED"",""Close"":7.210000038146973,""High"":7.239999771118164,""Low"":7.050000190734863,""Open"":7.050000190734863,""Volume"":6036.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-08"",""simbolo"":""ARCO"",""Close"":24059.728515625,""High"":24199.72693591542,""Low"":23459.735285808914,""Open"":23459.735285808914,""Volume"":4517.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-23"",""simbolo"":""AXPD"",""Close"":24.570354461669922,""High"":25.3459470987117,""Low"":24.510694073149647,""Open"":25.3459470987117,""Volume"":226.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-15"",""simbolo"":""BA"",""Close"":15650.0,""High"":15730.0,""Low"":15350.0,""Open"":15350.0,""Volume"":27747.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-15"",""simbolo"":""BABAD"",""Close"":19.419185638427734,""High"":19.627353641159253,""Low"":18.834329977347437,""Open"":19.329970239909212,""Volume"":4212.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-19"",""simbolo"":""BBD"",""Close"":5254.36767578125,""High"":5364.354439689136,""Low"":5179.37670038951,""Open"":5364.354439689136,""Volume"":20105.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-26"",""simbolo"":""BBV"",""Close"":37329.92578125,""High"":37446.52107260216,""Low"":36902.409712958746,""Open"":37019.00500431091,""Volume"":348.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-27"",""simbolo"":""BMNR"",""Close"":5465.0,""High"":5475.0,""Low"":5270.0,""Open"":5310.0,""Volume"":1897.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-02"",""simbolo"":""CADO"",""Close"":556.0,""High"":560.0,""Low"":519.0,""Open"":525.0,""Volume"":21068.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false
instrument,"{""date"":""2026-01-23"",""simbolo"":""CECO2"",""Close"":473.0,""High"":480.0,""Low"":463.0,""Open"":463.0,""Volume"":76528.0}",1,2026,1,2026-08-05T06:40:50.712Z,null,false


In [0]:
%sql
SELECT mes, count(*) FROM iol_challenge.bronze.raw_ingestion GROUP BY mes;


mes,count(*)
4,11423
7,12718
3,29209
5,11318
2,45465
8,1338
1,59418
6,12204
